# RFROM

The results are near-global ¼-degree x ¼-degree x 7-day resolution maps of OHCA for 58 different depth layers.

## File structure

**On ERDDAP:**

Netcdfs are unchunked. time: 5 or 4 7-day steps, mean_pressure: 58, latitude: 720, longitude: 1440. This means we cannot append files since time chunk sizes would vary. Also having all 58 depth levels in one chunk is not efficient since it is common to want individual depths.

**Plan for NODD**:

* Download netcdfs from ERDDAP to prep a netcdf for NODD
* Rewrite the netcdfs
* fix metadata if needed
* change the physical chunks to = (100, 1, 180, 180) ≈ 13 MB
* each file will be 100 time steps, all 58 levels, global. So 20-30 GB
* push to NODD in netcdf directory

## Setting the file names 

And cross-walking to the needed files that are on ERDDAP.

Files are here: https://data.pmel.noaa.gov/pmel/erddap/files/argo_rfromv23_temp/

metadata here: https://data.pmel.noaa.gov/pmel/erddap/info/argo_rfromv23_temp/index.html



In [2]:
import xarray as xr
ds = xr.open_dataset("/home/jovyan/shared-public/RFROMV23_TEMP_STABLE_1994_01.nc", chunks={})
ds

<xarray.Dataset> Size: 962MB
Dimensions:             (time: 4, mean_pressure: 58, latitude: 720,
                         longitude: 1440, vertices: 2)
Coordinates:
  * time                (time) datetime64[ns] 32B 1994-01-07 ... 1994-01-28
  * mean_pressure       (mean_pressure) float32 232B 2.5 10.0 ... 1.975e+03
  * latitude            (latitude) float32 3kB -89.88 -89.62 ... 89.62 89.88
  * longitude           (longitude) float32 6kB 0.125 0.375 ... 359.6 359.9
Dimensions without coordinates: vertices
Data variables:
    ocean_temperature   (time, mean_pressure, latitude, longitude) float32 962MB dask.array<chunksize=(4, 58, 720, 1440), meta=np.ndarray>
    mean_pressure_bnds  (mean_pressure, vertices) float32 464B dask.array<chunksize=(58, 2), meta=np.ndarray>
Attributes:
    Conventions:  CF-1.8
    title:        RFROM v2.3
    institution:  NOAA PMEL, CIMAR
    source:       Argo float data, EN4.2.2, CMEMES SSH, NOAA OI SST V2
    references:   Lyman, J.M. and G.C. Johnson. 2026. High Resolution Random ...
    history:      02/11/2026 v2.3
    comment:      none

In [8]:
url = (
    "https://data.pmel.noaa.gov/pmel/erddap/griddap/"
    "argo_rfromv23_temp.csv?time"
)
times = pd.read_csv(url)
times

,time
0,UTC
1,1993-01-01T00:00:00Z
2,1993-01-08T00:00:00Z
3,1993-01-15T00:00:00Z
4,1993-01-22T00:00:00Z
...,...
1666,2024-11-29T00:00:00Z
1667,2024-12-06T00:00:00Z
1668,2024-12-13T00:00:00Z
1669,2024-12-20T00:00:00Z


In [12]:
import pandas as pd

BASE_URL = (
    "https://data.pmel.noaa.gov/pmel/erddap/files/"
    "argo_rfromv23_temp"
)

url = (
    "https://data.pmel.noaa.gov/pmel/erddap/griddap/"
    "argo_rfromv23_temp.csv?time"
)
times = pd.read_csv(url, skiprows=[1])
times["time"] = pd.to_datetime(times["time"])

def make_file_blocks(times, block_size=100):
    times = pd.DatetimeIndex(times["time"])

    blocks = []

    for i in range(0, len(times), block_size):
        block_times = times[i:i + block_size]
        months = block_times.to_period("M").unique()

        urls = [
            (
                f"{BASE_URL}/"
                f"RFROMV23_TEMP_STABLE_{m.year}_{m.month:02d}.nc"
            )
            for m in months
        ]

        start = block_times[0]
        end = block_times[-1]

        filename = (
            f"RFROMV23_TEMP_STABLE_"
            f"{start:%Y-%m-%d}_{end:%Y-%m-%d}.nc"
        )

        blocks.append({
            "block": i // block_size,
            "times": block_times,
            "start": start,
            "end": end,
            "filename": filename,
            "urls": urls,
        })

    return blocks

In [13]:
blocks = make_file_blocks(times)
blocks[0]

/tmp/ipykernel_1994/1823271236.py:22: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  months = block_times.to_period("M").unique()


{'block': 0,
 'times': DatetimeIndex(['1993-01-01 00:00:00+00:00', '1993-01-08 00:00:00+00:00',
                '1993-01-15 00:00:00+00:00', '1993-01-22 00:00:00+00:00',
                '1993-01-29 00:00:00+00:00', '1993-02-05 00:00:00+00:00',
                '1993-02-12 00:00:00+00:00', '1993-02-19 00:00:00+00:00',
                '1993-02-26 00:00:00+00:00', '1993-03-05 00:00:00+00:00',
                '1993-03-12 00:00:00+00:00', '1993-03-19 00:00:00+00:00',
                '1993-03-26 00:00:00+00:00', '1993-04-02 00:00:00+00:00',
                '1993-04-09 00:00:00+00:00', '1993-04-16 00:00:00+00:00',
                '1993-04-23 00:00:00+00:00', '1993-04-30 00:00:00+00:00',
                '1993-05-07 00:00:00+00:00', '1993-05-14 00:00:00+00:00',
                '1993-05-21 00:00:00+00:00', '1993-05-28 00:00:00+00:00',
                '1993-06-04 00:00:00+00:00', '1993-06-11 00:00:00+00:00',
                '1993-06-18 00:00:00+00:00', '1993-06-25 00:00:00+00:00',
                

In [11]:
block_starts = times["time"].iloc[::100]
block_starts

0      1993-01-01 00:00:00+00:00
100    1994-12-02 00:00:00+00:00
200    1996-11-01 00:00:00+00:00
300    1998-10-02 00:00:00+00:00
400    2000-09-01 00:00:00+00:00
500    2002-08-02 00:00:00+00:00
600    2004-07-02 00:00:00+00:00
700    2006-06-02 00:00:00+00:00
800    2008-05-02 00:00:00+00:00
900    2010-04-02 00:00:00+00:00
1000   2012-03-02 00:00:00+00:00
1100   2014-01-31 00:00:00+00:00
1200   2016-01-01 00:00:00+00:00
1300   2017-12-01 00:00:00+00:00
1400   2019-11-01 00:00:00+00:00
1500   2021-10-01 00:00:00+00:00
1600   2023-09-01 00:00:00+00:00
Name: time, dtype: datetime64[ns, UTC]

In [14]:
for block in blocks:
    print(block["filename"])

RFROMV23_TEMP_STABLE_1993-01-01_1994-11-25.nc
RFROMV23_TEMP_STABLE_1994-12-02_1996-10-25.nc
RFROMV23_TEMP_STABLE_1996-11-01_1998-09-25.nc
RFROMV23_TEMP_STABLE_1998-10-02_2000-08-25.nc
RFROMV23_TEMP_STABLE_2000-09-01_2002-07-26.nc
RFROMV23_TEMP_STABLE_2002-08-02_2004-06-25.nc
RFROMV23_TEMP_STABLE_2004-07-02_2006-05-26.nc
RFROMV23_TEMP_STABLE_2006-06-02_2008-04-25.nc
RFROMV23_TEMP_STABLE_2008-05-02_2010-03-26.nc
RFROMV23_TEMP_STABLE_2010-04-02_2012-02-24.nc
RFROMV23_TEMP_STABLE_2012-03-02_2014-01-24.nc
RFROMV23_TEMP_STABLE_2014-01-31_2015-12-25.nc
RFROMV23_TEMP_STABLE_2016-01-01_2017-11-24.nc
RFROMV23_TEMP_STABLE_2017-12-01_2019-10-25.nc
RFROMV23_TEMP_STABLE_2019-11-01_2021-09-24.nc
RFROMV23_TEMP_STABLE_2021-10-01_2023-08-25.nc
RFROMV23_TEMP_STABLE_2023-09-01_2024-12-27.nc
